In [1]:
pip install tensorflow numpy matplotlib opencv-python scikit-learn


####Mount path

In [2]:
from google.colab import drive

# Mount your Google Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [3]:
import os

# Check your Google Drive contents
drive_path = '/content/drive/My Drive/'  # Mounted drive root directory

# List contents of the drive root or specific folders to find your directories
os.listdir(drive_path)  # This will list top-level contents

['2)Bioinformatics_work',
 'my_projects',
 '1)Molecular_collection',
 'phd_preparation',
 'Bulk cell transcriptomic data analysis.gdoc',
 'nr14_old_drive',
 'COLLECTIONS',
 'Colab Notebooks',
 'my_work',
 'PYTHON']

##Load data

In [4]:
adenocarcinoma_traning = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Adenocarcinoma_colorectal_taining'
normal_training = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Normal_training'

adenocarcinoma_test = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/adenocarcinoma_test'
normal_test = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/norma_test'


##Visualization

###1_adenocarcinoma_training:

In [5]:
import os
import matplotlib.pyplot as plt

# Adenocarcinoma TRAINING
adenocarcinoma_traning = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Adenocarcinoma_colorectal_taining'
plt.figure(figsize=(22, 12))

for i, f in enumerate(sorted(os.listdir(adenocarcinoma_traning))[:15]):
    plt.subplot(3, 5, i+1, xticks=[], yticks=[])
    img = plt.imread(os.path.join(adenocarcinoma_traning, f))
    plt.imshow(img)
    plt.title(f"Adenoc Training{i+1}", fontsize=14, pad=4, color='red')  # Red title
plt.suptitle("Adenocarcinoma TRAINING", y=0.98, fontsize=18, color='red', weight='bold')  # Red main title
plt.tight_layout(w_pad=0.02, h_pad=0.04)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

###2_Normal_TRAINING:

In [6]:
# Normal TRAINING
normal_training = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Normal_training'
plt.figure(figsize=(22, 12))

for i, f in enumerate(sorted(os.listdir(normal_training))[:15]):
    plt.subplot(3, 5, i+1, xticks=[], yticks=[])
    img = plt.imread(os.path.join(normal_training, f))
    plt.imshow(img)
    plt.title(f"Normal_Training{i+1}", fontsize=14, pad=4, color='green')
plt.suptitle("Normal TRAINING", y=0.98, fontsize=18, color='green', weight='bold')
plt.tight_layout(w_pad=0.02, h_pad=0.04)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

###3_Adenocarcinoma_TEST

In [7]:
# Adenocarcinoma TEST
adenocarcinoma_test = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/adenocarcinoma_test'
plt.figure(figsize=(22, 12))

for i, f in enumerate(sorted(os.listdir(adenocarcinoma_test))[:15]):
    plt.subplot(3, 5, i+1, xticks=[], yticks=[])
    img = plt.imread(os.path.join(adenocarcinoma_test, f))
    plt.imshow(img)
    plt.title(f"Adenoc TEST{i+1}", fontsize=14, pad=4, color='blue')
plt.suptitle("Adenocarcinoma TEST", y=0.98, fontsize=18, color='blue', weight='bold')
plt.tight_layout(w_pad=0.02, h_pad=0.04)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

In [8]:
# Normal TEST
normal_test = '/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/norma_test'
plt.figure(figsize=(22, 12))

for i, f in enumerate(sorted(os.listdir(normal_test))[:15]):
    plt.subplot(3, 5, i+1, xticks=[], yticks=[])
    img = plt.imread(os.path.join(normal_test, f))
    plt.imshow(img)
    plt.title(f"Normal_TEST{i+1}", fontsize=14, pad=4, color='purple')
plt.suptitle("Normal TEST", y=0.98, fontsize=18, color='purple', weight='bold')
plt.tight_layout(w_pad=0.02, h_pad=0.04)
plt.show()

Output hidden; open in https://colab.research.google.com to view.

#**Data** pre-process

In [9]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt


####Define preprocessing functions

In [10]:
import cv2
import os
import numpy as np
import matplotlib.pyplot as plt

IMG_SIZE = 128  # Standard size for resizing

# Function to resize images
def resize_image(img, size=IMG_SIZE):
    return cv2.resize(img, (size, size))

# Function to apply Gaussian blur (for noise reduction)
def apply_gaussian_blur(img):
    return cv2.GaussianBlur(img, (3, 3), 0)

# Function to apply Contrast Enhancement (CLAHE in LAB color space)
def enhance_contrast_color(img):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    l = clahe.apply(l)
    lab = cv2.merge((l, a, b))
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)

# Function to apply Edge Detection overlay
def apply_edge_overlay(img, alpha=0.6):
    edges = cv2.Canny(img, 50, 150)  # Detect edges
    edges_colored = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)  # Convert to 3-channel
    return cv2.addWeighted(img, alpha, edges_colored, 1 - alpha, 0)  # Blend images

# Function to preprocess ALL images in a folder
def load_and_preprocess_all(folder):
    processed_images = []
    filenames = sorted(os.listdir(folder))  # Get all image filenames

    for filename in filenames:
        img_path = os.path.join(folder, filename)
        img = cv2.imread(img_path)  # Load in color
        if img is not None:
            img_resized = resize_image(img)
            img_blur = apply_gaussian_blur(img_resized)
            img_contrast = enhance_contrast_color(img_blur)
            img_edge_overlay = apply_edge_overlay(img_contrast)  # Edge overlay
            processed_images.append((filename, img_edge_overlay))  # Store all images
    return processed_images



###1) PRE-PROCESS adenocarcinoma Training images

##Save all processed images

In [11]:
# Define input folders
input_folders = {
    "adenocarcinoma_traning": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Adenocarcinoma_colorectal_taining",
    "normal_training": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Normal_training",
    "adenocarcinoma_test": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/adenocarcinoma_test",
    "normal_test": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/norma_test"
}



# Define output folders (Keys must exactly match input_folders)
output_folders = {
    "adenocarcinoma_traning": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_train/",
    "normal_training": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_train/",
    "adenocarcinoma_test": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_test/",
    "normal_test": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_test/"
}


# Function to save all processed images
def save_preprocessed_images(images, save_folder):
    if not os.path.exists(save_folder):
        os.makedirs(save_folder)

    for filename, processed_img in images:
        save_path = os.path.join(save_folder, filename)
        cv2.imwrite(save_path, processed_img)  # Save image

# Process and save images for all four categories
for category, input_folder in input_folders.items():
    print(f"Processing images for: {category}...")
    processed_images = load_and_preprocess_all(input_folder)
    save_preprocessed_images(processed_images, output_folders[category])
    print(f"✅ Saved {len(processed_images)} images in {output_folders[category]}")


Processing images for: adenocarcinoma_traning...
✅ Saved 700 images in /content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_train/
Processing images for: normal_training...
✅ Saved 58 images in /content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_train/
Processing images for: adenocarcinoma_test...
✅ Saved 95 images in /content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_test/
Processing images for: normal_test...
✅ Saved 18 images in /content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_test/


In [12]:
import os

# Define input and output folders
input_folders = {
    "adenocarcinoma_traning": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Adenocarcinoma_colorectal_taining",
    "normal_training": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/training/Normal_training",
    "adenocarcinoma_test": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/adenocarcinoma_test",
    "normal_test": "/content/drive/My Drive/PYTHON/ML_image_process/binary_classification_adenocarcinoma/data/test/norma_test"
}

output_folders = {
    "adenocarcinoma_traning": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_train/",
    "normal_training": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_train/",
    "adenocarcinoma_test": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_adenocarcinoma_test/",
    "normal_test": "/content/drive/My Drive/PYTHON/ML_image_process/preprocessed/prep_normal_test/"
}

# Function to count images in each folder
def count_images(folder_path):
    return len([f for f in os.listdir(folder_path) if f.endswith(('.png', '.jpg', '.jpeg', '.tif', '.bmp'))])

# Count images in input folders
print("Input Folders:")
for name, path in input_folders.items():
    if os.path.exists(path):
        print(f"{name}: {count_images(path)} images")
    else:
        print(f"{name}: Folder not found")

# Count images in output folders
print("\nOutput Folders:")
for name, path in output_folders.items():
    if os.path.exists(path):
        print(f"{name}: {count_images(path)} images")
    else:
        print(f"{name}: Folder not found")


Input Folders:
adenocarcinoma_traning: 700 images
normal_training: 58 images
adenocarcinoma_test: 95 images
normal_test: 18 images

Output Folders:
adenocarcinoma_traning: 700 images
normal_training: 58 images
adenocarcinoma_test: 95 images
normal_test: 18 images
